# ArchiGen V3: learned material references and semantic feedback

Private academic experiment. SD1.5 + ten-channel ControlNet (seven semantic + RGB material reference), initialized from selected V2 step 2000 and its existing rank-8 UNet LoRA.

Recover the checksummed previous teacher using fixed inverse CE-class-cost bias correction; evaluate all validation photos before reuse. If needed, fine-tune using unweighted CE plus explicit wall hard negatives. Require validation mIoU >= .30, opening IoU >= .35, door IoU >= .10 and wall false-opening rate <= .05. A failed gate stops full diffusion training. Smoke deliberately bypasses quality gating and never produces deployable weights.

Diffusion uses the standard opening-weighted Min-SNR objective plus low-noise, differentiable semantic feedback, outside-opening penalties and a small material-statistics term. Include real annotated wall-only crops; never erase openings from labels. Preserve existing CMP splits and IRF building groups. Four architectural style classes are not supervised.

Validation discloses target-derived reference crops separately from independent training-building references. Do not report mismatched-material LPIPS. Teacher scores are training-reward diagnostics, not independent accuracy. Tests remain untouched; no automatic promotion.

Run smoke first, then full mode: the corrected teacher (or up to 12 further precision-focused epochs) and at most 1500 diffusion optimizer steps, physical batch 1 / effective batch 8, one GPU, mixed precision, gradient checkpointing and 8-bit optimizer. The trained export retains laptop CPU-offloaded inference without the teacher.


In [ ]:
import os, sys, subprocess
from pathlib import Path
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['PYTHONUNBUFFERED'] = '1'
PINNED_COMMIT = ""
MODE = "smoke"
BRANCH = 'codex/facade-material-controlnet-v3'
REPO = Path('/kaggle/working/archigen-source')
if not REPO.exists():
    subprocess.run(['git','clone','--depth','1','--single-branch','--branch',BRANCH,'https://github.com/ayebkalil/archigen-3d.git',str(REPO)],check=True)
if not PINNED_COMMIT:
    raise ValueError('Stage this notebook with manage stage-kernel to pin a Git SHA')
subprocess.run(['git','fetch','--depth','1','origin',PINNED_COMMIT],cwd=REPO,check=True)
subprocess.run(['git','checkout','--detach',PINNED_COMMIT],cwd=REPO,check=True)
print('Source commit:', PINNED_COMMIT, flush=True)
# Retain Kaggle native Torch/CUDA; only install model/tracking dependencies.
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(REPO/'experiments/facade_material_v3/requirements.txt')],check=True)


In [ ]:
# Optional remote tracking / HF authentication belong in Kaggle Secrets.
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    for key in ('HF_TOKEN','MLFLOW_TRACKING_URI','MLFLOW_TRACKING_TOKEN','MLFLOW_TRACKING_USERNAME','MLFLOW_TRACKING_PASSWORD'):
        try:
            value = secrets.get_secret(key)
            if value: os.environ[key] = value
        except Exception:
            pass
except ImportError:
    pass
from urllib.parse import urlsplit
if os.getenv('MLFLOW_TRACKING_URI') and urlsplit(os.environ['MLFLOW_TRACKING_URI']).hostname in {'localhost','127.0.0.1'}:
    raise ValueError('PC localhost is not reachable from Kaggle; remove URI for portable SQLite tracking')
subprocess.run([sys.executable,'-m','experiments.facade_material_v3.kaggle_run','--mode',MODE],cwd=REPO,check=True)


## Output and local use
Download `archigen-facade-material-v3-model-and-mlflow.zip`. It contains checksummed material10 ControlNet + LoRA, teacher provenance/weights for research, validation comparisons, requirements and portable MLflow events for both runs.

Use `python -m experiments.facade_material_v3.manage install --archive PATH.zip --destination NEW_FOLDER --tracking-uri sqlite:///mlflow.db` to validate/install and import histories. Smoke exports are rejected. Existing model folders and production are never overwritten.

`MaterialFacadeRuntime.generate_material(semantic7, material_reference_512)` uses the pinned SD1.5 base, sequential CPU offload and FP16. The teacher is not required at inference. New weights still require profiling on the RTX3050 and multi-house visual tests before release.
